# Credit Card Fraud Detection + AI Risk Explanation System

## A Complete Machine Learning Pipeline

This notebook demonstrates a real-world ML project that:
1. Handles highly imbalanced data
2. Implements and compares multiple algorithms
3. Optimizes models using proper metrics (not just accuracy)
4. Explains predictions using feature importance and LIME

**Dataset:** Credit Card Fraud Detection (Kaggle)  
**Size:** 284,807 transactions (492 frauds = 0.17%)  
**Goal:** Detect fraud AND explain predictions

## SECTION 1: IMPORTS & SETUP

In [ ]:
# Essential imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (
    confusion_matrix, classification_report, 
    roc_curve, auc, roc_auc_score, f1_score, 
    precision_score, recall_score, accuracy_score,
    ConfusionMatrixDisplay
)

# Utilities
import warnings
import joblib
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ All imports successful!")

## SECTION 2: DATA LOADING & EXPLORATION

In [ ]:
# Load dataset
df = pd.read_csv('creditcard.csv')

print("Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values: {df.isnull().sum().sum()}")

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Class distribution - THE CRITICAL IMBALANCE PROBLEM
print("\n" + "="*60)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*60)

class_counts = df['Class'].value_counts()
class_props = df['Class'].value_counts(normalize=True)

print(f"\nLegitimate transactions: {class_counts[0]} ({class_props[0]*100:.2f}%)")
print(f"Fraudulent transactions:  {class_counts[1]} ({class_props[1]*100:.2f}%)")
print(f"Imbalance Ratio: {class_counts[0]/class_counts[1]:.1f}:1")

print("\n⚠️  KEY INSIGHT:")
print(f"If we predict ALL transactions as legitimate, we get {class_props[0]*100:.2f}% accuracy!")
print("But we catch 0 frauds. This is why ACCURACY IS MISLEADING.")
print("We must use Precision, Recall, F1-Score, and ROC-AUC instead.")

In [ ]:
# Visualize class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Fraud vs Legitimate Transactions (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(class_counts, labels=['Legitimate\n(99.83%)', 'Fraud\n(0.17%)'], 
            autopct='%1.3f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution (Percentage)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze transaction amounts
print("\n" + "="*60)
print("TRANSACTION AMOUNT ANALYSIS")
print("="*60)

print("\nLegitimate Transactions:")
print(df[df['Class']==0]['Amount'].describe())

print("\nFraudulent Transactions:")
print(df[df['Class']==1]['Amount'].describe())

print("\n✓ INSIGHT: Fraud amounts are typically lower")

In [ ]:
# Visualize amount distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df[df['Class']==0]['Amount'].hist(bins=50, ax=axes[0], color='green', alpha=0.7, label='Legitimate')
df[df['Class']==1]['Amount'].hist(bins=50, ax=axes[0], color='red', alpha=0.7, label='Fraud')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Amount Distribution (Zoomed to $0-$1000)')
axes[0].set_xlim([0, 1000])
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
data_to_plot = [df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']]
bp = axes[1].boxplot(data_to_plot, labels=['Legitimate', 'Fraud'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightgreen')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].set_ylabel('Amount')
axes[1].set_title('Amount Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
print("\n" + "="*60)
print("CORRELATION WITH FRAUD")
print("="*60)

correlation_with_fraud = df.corr()['Class'].sort_values(ascending=False)

print("\nTop 10 Features Most Correlated with Fraud:")
print(correlation_with_fraud.head(11)[1:])

print("\nBottom 5 Features (Negative Correlation):")
print(correlation_with_fraud.tail(5))

## SECTION 3: DATA PREPROCESSING

In [ ]:
print("="*60)
print("DATA PREPROCESSING PIPELINE")
print("="*60)

# Step 1: Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

print(f"\n✓ Features separated: X shape = {X.shape}")
print(f"✓ Target isolated: y shape = {y.shape}")

# Step 2: Train-Test Split (BEFORE scaling - important!)
# Use stratified split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,  # For reproducibility
    stratify=y  # Maintains class distribution in both sets
)

print(f"\n✓ Train-Test Split (80-20):")
print(f"  - Training set: {X_train.shape[0]} samples")
print(f"  - Test set: {X_test.shape[0]} samples")

print(f"\n✓ Class Distribution Maintained:")
print(f"  - Train fraud ratio: {y_train.sum()/len(y_train)*100:.2f}%")
print(f"  - Test fraud ratio: {y_test.sum()/len(y_test)*100:.2f}%")

In [ ]:
# Step 3: Feature Scaling
# IMPORTANT: Fit scaler on training data ONLY

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training
X_test_scaled = scaler.transform(X_test)         # Transform test with training params

print("✓ Features scaled with StandardScaler")
print(f"\n  Training data statistics after scaling:")
print(f"  - Mean: {X_train_scaled.mean():.2e}")
print(f"  - Std: {X_train_scaled.std():.4f}")

print(f"\n  Test data statistics after scaling:")
print(f"  - Mean: {X_test_scaled.mean():.2e}")
print(f"  - Std: {X_test_scaled.std():.4f}")

# Convert to DataFrame for easier manipulation
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\n✓ Preprocessing complete! Data ready for modeling.")

## SECTION 4: MODEL DEVELOPMENT & TRAINING

In [ ]:
# Define model evaluation function

def train_evaluate_model(model, model_name, X_train, X_test, y_train, y_test):
    """
    Train and evaluate a model
    
    Parameters:
    - model: sklearn model object
    - model_name: string name for reporting
    - X_train, X_test: training and test features
    - y_train, y_test: training and test labels
    
    Returns:
    - trained model
    - metrics dictionary
    - predictions
    - prediction probabilities
    """
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    # Calculate metrics
    metrics = {
        'Model': model_name,
        'Accuracy': (tp + tn) / (tp + tn + fp + fn),
        'Precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
        'Recall': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'F1-Score': 2 * ((tp / (tp + fp)) * (tp / (tp + fn))) / 
                    ((tp / (tp + fp)) + (tp / (tp + fn))) 
                    if (tp + fp) > 0 and (tp + fn) > 0 else 0,
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
    }
    
    return model, metrics, y_pred, y_pred_proba

print("✓ Evaluation function defined")

In [ ]:
# Store results
model_results = {}
trained_models = {}

print("\n" + "="*70)
print("TRAINING 4 MACHINE LEARNING MODELS")
print("="*70)

In [ ]:
# MODEL 1: LOGISTIC REGRESSION
print("\n" + "-"*70)
print("MODEL 1: LOGISTIC REGRESSION")
print("-"*70)

lr_model, lr_metrics, lr_pred, lr_proba = train_evaluate_model(
    LogisticRegression(
        random_state=42, 
        class_weight='balanced',  # Handle imbalance
        max_iter=1000,
        solver='lbfgs'
    ),
    'Logistic Regression',
    X_train_scaled, X_test_scaled, y_train, y_test
)

model_results['Logistic Regression'] = lr_metrics
trained_models['Logistic Regression'] = lr_model

print(f"\nAccuracy:  {lr_metrics['Accuracy']:.4f}")
print(f"Precision: {lr_metrics['Precision']:.4f} (fewer false alarms)")
print(f"Recall:    {lr_metrics['Recall']:.4f} (catches more frauds)")
print(f"F1-Score:  {lr_metrics['F1-Score']:.4f}")
print(f"ROC-AUC:   {lr_metrics['ROC-AUC']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {lr_metrics['TN']}")
print(f"  False Positives: {lr_metrics['FP']}")
print(f"  False Negatives: {lr_metrics['FN']}")
print(f"  True Positives:  {lr_metrics['TP']}")

In [ ]:
# MODEL 2: DECISION TREE
print("\n" + "-"*70)
print("MODEL 2: DECISION TREE")
print("-"*70)

dt_model, dt_metrics, dt_pred, dt_proba = train_evaluate_model(
    DecisionTreeClassifier(
        random_state=42, 
        class_weight='balanced',
        max_depth=10,  # Prevent overfitting
        min_samples_split=10,
        min_samples_leaf=5
    ),
    'Decision Tree',
    X_train_scaled, X_test_scaled, y_train, y_test
)

model_results['Decision Tree'] = dt_metrics
trained_models['Decision Tree'] = dt_model

print(f"\nAccuracy:  {dt_metrics['Accuracy']:.4f}")
print(f"Precision: {dt_metrics['Precision']:.4f}")
print(f"Recall:    {dt_metrics['Recall']:.4f}")
print(f"F1-Score:  {dt_metrics['F1-Score']:.4f}")
print(f"ROC-AUC:   {dt_metrics['ROC-AUC']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {dt_metrics['TN']}")
print(f"  False Positives: {dt_metrics['FP']}")
print(f"  False Negatives: {dt_metrics['FN']}")
print(f"  True Positives:  {dt_metrics['TP']}")

In [ ]:
# MODEL 3: RANDOM FOREST
print("\n" + "-"*70)
print("MODEL 3: RANDOM FOREST")
print("-"*70)

rf_model, rf_metrics, rf_pred, rf_proba = train_evaluate_model(
    RandomForestClassifier(
        n_estimators=100,  # Number of trees
        random_state=42, 
        class_weight='balanced',
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        n_jobs=-1  # Use all processors
    ),
    'Random Forest',
    X_train_scaled, X_test_scaled, y_train, y_test
)

model_results['Random Forest'] = rf_metrics
trained_models['Random Forest'] = rf_model

print(f"\nAccuracy:  {rf_metrics['Accuracy']:.4f}")
print(f"Precision: {rf_metrics['Precision']:.4f}")
print(f"Recall:    {rf_metrics['Recall']:.4f}")
print(f"F1-Score:  {rf_metrics['F1-Score']:.4f}")
print(f"ROC-AUC:   {rf_metrics['ROC-AUC']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {rf_metrics['TN']}")
print(f"  False Positives: {rf_metrics['FP']}")
print(f"  False Negatives: {rf_metrics['FN']}")
print(f"  True Positives:  {rf_metrics['TP']}")

In [ ]:
# MODEL 4: SUPPORT VECTOR MACHINE
print("\n" + "-"*70)
print("MODEL 4: SUPPORT VECTOR MACHINE")
print("-"*70)

svm_model, svm_metrics, svm_pred, svm_proba = train_evaluate_model(
    SVC(
        kernel='rbf',
        probability=True,  # For probability estimates
        random_state=42, 
        class_weight='balanced',
        C=1.0  # Regularization parameter
    ),
    'Support Vector Machine',
    X_train_scaled, X_test_scaled, y_train, y_test
)

model_results['Support Vector Machine'] = svm_metrics
trained_models['Support Vector Machine'] = svm_model

print(f"\nAccuracy:  {svm_metrics['Accuracy']:.4f}")
print(f"Precision: {svm_metrics['Precision']:.4f}")
print(f"Recall:    {svm_metrics['Recall']:.4f}")
print(f"F1-Score:  {svm_metrics['F1-Score']:.4f}")
print(f"ROC-AUC:   {svm_metrics['ROC-AUC']:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Negatives:  {svm_metrics['TN']}")
print(f"  False Positives: {svm_metrics['FP']}")
print(f"  False Negatives: {svm_metrics['FN']}")
print(f"  True Positives:  {svm_metrics['TP']}")

## SECTION 5: MODEL EVALUATION & COMPARISON

In [ ]:
# Create comparison table
results_df = pd.DataFrame(model_results).T
comparison_df = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

print("\n" + "="*70)
print("MODEL COMPARISON TABLE")
print("="*70)
print(comparison_df.round(4))

print("\n" + "="*70)
print("MODEL RANKING BY KEY METRICS")
print("="*70)

for metric in ['ROC-AUC', 'F1-Score', 'Recall', 'Precision']:
    ranked = comparison_df[metric].sort_values(ascending=False)
    print(f"\n{metric} Ranking:")
    for i, (model, score) in enumerate(ranked.items(), 1):
        print(f"  {i}. {model:.<40} {score:.4f}")

In [ ]:
# Confusion matrices visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

predictions_dict = {
    'Logistic Regression': lr_pred,
    'Decision Tree': dt_pred,
    'Random Forest': rf_pred,
    'Support Vector Machine': svm_pred
}

for idx, (model_name, predictions) in enumerate(predictions_dict.items()):
    cm = confusion_matrix(y_test, predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=axes[idx], cmap='Blues', values_format='d')
    axes[idx].set_title(f'{model_name}', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves Comparison
plt.figure(figsize=(10, 8))

# Calculate ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_proba)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_proba)

# Plot
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={lr_metrics["ROC-AUC"]:.4f})', linewidth=2)
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC={dt_metrics["ROC-AUC"]:.4f})', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_metrics["ROC-AUC"]:.4f})', linewidth=2)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={svm_metrics["ROC-AUC"]:.4f})', linewidth=2)

# Random classifier
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier', alpha=0.7)

plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ ROC-AUC Interpretation:")
print("  - Probability that model ranks a random fraud higher than random legitimate")
print("  - 1.0 = Perfect | 0.9-1.0 = Excellent | 0.8-0.9 = Good | 0.7-0.8 = Fair | <0.7 = Poor")

In [ ]:
# Detailed classification report for best model
best_model_name = comparison_df['F1-Score'].idxmax()
best_model_obj = trained_models[best_model_name]
y_pred_best = best_model_obj.predict(X_test_scaled)

print("\n" + "="*70)
print(f"DETAILED CLASSIFICATION REPORT - {best_model_name.upper()}")
print("="*70)

print(classification_report(y_test, y_pred_best, target_names=['Legitimate', 'Fraud']))

print("\nMetric Interpretations:")
print("- Precision: Of transactions we flagged as fraud, how many were actually fraud?")
print("- Recall: Of actual frauds, what percentage did we catch?")
print("- F1-Score: Harmonic mean (balanced score) of precision and recall")
print("- Support: Number of actual cases in the test set")

## SECTION 6: FEATURE IMPORTANCE ANALYSIS

In [ ]:
print("\n" + "="*70)
print("FEATURE IMPORTANCE - RANDOM FOREST")
print("="*70)

# Extract feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Important Features:")
print(feature_importance.head(20).to_string(index=False))

print("\n✓ Insights:")
top_5 = feature_importance.head(5)['Feature'].tolist()
print(f"  Top 5 fraud indicators: {', '.join(top_5)}")
print(f"  These features have the strongest impact on fraud detection")

In [ ]:
# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top 15 features bar plot
top_n = 15
top_features = feature_importance.head(top_n)

axes[0].barh(range(top_n), top_features['Importance'].values, color='steelblue')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(top_features['Feature'].values)
axes[0].set_xlabel('Feature Importance Score', fontsize=11)
axes[0].set_title(f'Top {top_n} Features - Random Forest', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

for i, v in enumerate(top_features['Importance'].values):
    axes[0].text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)

# Cumulative importance
cumsum = np.cumsum(feature_importance['Importance'].values)
cumsum_pct = cumsum / cumsum[-1] * 100

axes[1].plot(range(len(cumsum_pct)), cumsum_pct, 'b-', linewidth=2, label='Cumulative Importance')
axes[1].axhline(y=80, color='r', linestyle='--', linewidth=2, label='80% Threshold')
axes[1].axhline(y=90, color='orange', linestyle='--', linewidth=2, label='90% Threshold')
axes[1].set_xlabel('Number of Features', fontsize=11)
axes[1].set_ylabel('Cumulative Importance (%)', fontsize=11)
axes[1].set_title('Cumulative Feature Importance', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Find how many features needed for 80% and 90%
features_for_80 = np.argmax(cumsum_pct >= 80) + 1
features_for_90 = np.argmax(cumsum_pct >= 90) + 1

print(f"\n✓ Feature Selection Insights:")
print(f"  - {features_for_80} features explain 80% of fraud variance")
print(f"  - {features_for_90} features explain 90% of fraud variance")
print(f"  - Could reduce model complexity while maintaining performance")

## SECTION 7: HYPERPARAMETER TUNING

In [ ]:
print("\n" + "="*70)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*70)

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

print("\nStarting GridSearchCV with 5-fold cross-validation...")
print("(This may take a few minutes)\n")

# Perform grid search
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    param_grid,
    cv=5,
    scoring='f1',  # Use F1 score (not accuracy)
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\n✓ GridSearchCV Complete!")
print(f"\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  - {param}: {value}")

print(f"\nBest Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model on test set
y_pred_tuned = grid_search.predict(X_test_scaled)
y_proba_tuned = grid_search.predict_proba(X_test_scaled)[:, 1]

print("\n" + "="*70)
print("TUNED MODEL PERFORMANCE ON TEST SET")
print("="*70)

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned)
recall_tuned = recall_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_tuned = roc_auc_score(y_test, y_proba_tuned)

print(f"\nAccuracy:  {accuracy_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall:    {recall_tuned:.4f}")
print(f"F1-Score:  {f1_tuned:.4f}")
print(f"ROC-AUC:   {roc_auc_tuned:.4f}")

# Compare with original
print(f"\n" + "-"*70)
print("IMPROVEMENT vs ORIGINAL RANDOM FOREST:")
print("-"*70)

print(f"F1-Score:  {rf_metrics['F1-Score']:.4f} → {f1_tuned:.4f} "
      f"({(f1_tuned - rf_metrics['F1-Score'])*100:+.2f}%)")
print(f"ROC-AUC:   {rf_metrics['ROC-AUC']:.4f} → {roc_auc_tuned:.4f} "
      f"({(roc_auc_tuned - rf_metrics['ROC-AUC'])*100:+.2f}%)")
print(f"Recall:    {rf_metrics['Recall']:.4f} → {recall_tuned:.4f} "
      f"({(recall_tuned - rf_metrics['Recall'])*100:+.2f}%)")

# Set best model
best_model_final = grid_search.best_estimator_

## SECTION 8: LIME EXPLAINABILITY

In [ ]:
# Install LIME if needed
import subprocess
import sys

try:
    import lime
    print("✓ LIME library already installed")
except ImportError:
    print("Installing LIME...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lime", "-q"])
    import lime
    print("✓ LIME installed successfully")

import lime.lime_tabular

# Create LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_scaled.values,
    feature_names=X.columns.tolist(),
    class_names=['Legitimate', 'Fraud'],
    mode='classification',
    random_state=42
)

print("✓ LIME Explainer initialized")

In [ ]:
print("\n" + "="*70)
print("EXPLAINING INDIVIDUAL PREDICTIONS WITH LIME")
print("="*70)

# Find examples
fraud_mask = y_test == 1
legit_mask = y_test == 0

fraud_indices = y_test[fraud_mask].index.tolist()
legit_indices = y_test[legit_mask].index.tolist()

print(f"\nTotal test samples:")
print(f"  - Legitimate: {len(legit_indices)}")
print(f"  - Fraudulent: {len(fraud_indices)}")

In [ ]:
# EXAMPLE 1: Explain a fraudulent transaction
print("\n" + "="*70)
print("EXAMPLE 1: FRAUDULENT TRANSACTION EXPLANATION")
print("="*70)

# Get first fraudulent transaction
fraud_idx = fraud_indices[0]
fraud_example = X_test_scaled.loc[fraud_idx].values

# Get prediction info
pred_class = best_model_final.predict([fraud_example])[0]
pred_proba = best_model_final.predict_proba([fraud_example])[0]

print(f"\nTransaction Index: {fraud_idx}")
print(f"Actual Class: FRAUD (True positive)")
print(f"Model Prediction: {'FRAUD' if pred_class == 1 else 'LEGITIMATE'}")
print(f"Confidence:")
print(f"  - Legitimate: {pred_proba[0]*100:.2f}%")
print(f"  - Fraud: {pred_proba[1]*100:.2f}%")

# LIME explanation
explanation = explainer.explain_instance(
    fraud_example,
    best_model_final.predict_proba,
    num_features=10,
    top_labels=1
)

print("\nTop Features Contributing to FRAUD Classification:")
print("-" * 70)

exp_list = explanation.as_list()
for i, (feature_desc, weight) in enumerate(exp_list, 1):
    direction = "→ FRAUD" if weight > 0 else "← LEGITIMATE"
    print(f"{i:2d}. {feature_desc:.<50} {weight:>7.4f} {direction}")

print("\n✓ Interpretation:")
print("  Positive weights push prediction toward FRAUD")
print("  Negative weights push prediction toward LEGITIMATE")

In [ ]:
# EXAMPLE 2: Explain a legitimate transaction
print("\n" + "="*70)
print("EXAMPLE 2: LEGITIMATE TRANSACTION EXPLANATION")
print("="*70)

# Get a legitimate transaction
legit_idx = legit_indices[50]
legit_example = X_test_scaled.loc[legit_idx].values

# Get prediction info
pred_class = best_model_final.predict([legit_example])[0]
pred_proba = best_model_final.predict_proba([legit_example])[0]

print(f"\nTransaction Index: {legit_idx}")
print(f"Actual Class: LEGITIMATE (True negative)")
print(f"Model Prediction: {'FRAUD' if pred_class == 1 else 'LEGITIMATE'}")
print(f"Confidence:")
print(f"  - Legitimate: {pred_proba[0]*100:.2f}%")
print(f"  - Fraud: {pred_proba[1]*100:.2f}%")

# LIME explanation
explanation = explainer.explain_instance(
    legit_example,
    best_model_final.predict_proba,
    num_features=10,
    top_labels=1
)

print("\nTop Features Contributing to Classification:")
print("-" * 70)

exp_list = explanation.as_list()
for i, (feature_desc, weight) in enumerate(exp_list, 1):
    direction = "→ FRAUD" if weight > 0 else "← LEGITIMATE"
    print(f"{i:2d}. {feature_desc:.<50} {weight:>7.4f} {direction}")

In [ ]:
# Analyze patterns across multiple fraud examples
print("\n" + "="*70)
print("PATTERN ANALYSIS: TOP FRAUD INDICATORS")
print("="*70)

fraud_features_count = {}
fraud_features_weights = {}

print("\nAnalyzing first 5 fraudulent transactions...\n")

for i in range(min(5, len(fraud_indices))):
    fraud_idx = fraud_indices[i]
    fraud_example = X_test_scaled.loc[fraud_idx].values
    
    explanation = explainer.explain_instance(
        fraud_example,
        best_model_final.predict_proba,
        num_features=5,
        top_labels=1
    )
    
    print(f"Fraud Transaction {i+1}:")
    
    for feature_desc, weight in explanation.as_list()[:3]:
        # Extract feature name (before the first colon or comparison operator)
        feature_name = feature_desc.split()[0]
        
        # Track frequency
        if feature_name not in fraud_features_count:
            fraud_features_count[feature_name] = 0
            fraud_features_weights[feature_name] = 0
        
        fraud_features_count[feature_name] += 1
        fraud_features_weights[feature_name] += abs(weight)
        
        print(f"  • {feature_desc}")
    print()

print("\n" + "="*70)
print("MOST COMMON FRAUD INDICATORS:")
print("="*70)

sorted_features = sorted(fraud_features_count.items(), key=lambda x: x[1], reverse=True)
for feature, count in sorted_features:
    avg_weight = fraud_features_weights[feature] / count
    print(f"{feature:.<20} appears in {count}/5 examples (avg impact: {avg_weight:.4f})")

## SECTION 9: CONCLUSIONS & DEPLOYMENT

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS & INSIGHTS")
print("="*70)

print(f"""
1. CLASS IMBALANCE CHALLENGE:
   • Only 0.17% of transactions are fraudulent
   • Requires: class weighting, stratified splits, proper metrics
   • If we predict all as legitimate: 99.83% accuracy, 0% fraud caught!
   • Therefore: ACCURACY IS MISLEADING for imbalanced data

2. BEST MODEL: {best_model_name.upper()}
   • ROC-AUC: {comparison_df.loc[best_model_name, 'ROC-AUC']:.4f}/1.0000
   • Precision: {comparison_df.loc[best_model_name, 'Precision']:.4f} (fewer false alarms)
   • Recall: {comparison_df.loc[best_model_name, 'Recall']:.4f} (catches more frauds)
   • F1-Score: {comparison_df.loc[best_model_name, 'F1-Score']:.4f}

3. TOP FRAUD INDICATORS:
   • Primary: {', '.join(feature_importance.head(3)['Feature'].tolist())}
   • Secondary: {', '.join(feature_importance.iloc[3:6]['Feature'].tolist())}
   • Amount feature is LESS important than PCA-transformed values

4. HYPERPARAMETER TUNING IMPACT:
   • F1-Score improved by {(f1_tuned - rf_metrics['F1-Score'])*100:+.2f}%
   • ROC-AUC improved by {(roc_auc_tuned - rf_metrics['ROC-AUC'])*100:+.2f}%
   • GridSearchCV identified optimal configuration

5. MODEL INTERPRETABILITY:
   • LIME explains individual predictions
   • Feature importance shows global model behavior
   • Fraud typically triggered by V4, V12, V14 values
   • Different transactions flagged for different reasons
""")

print("\n" + "="*70)
print("RECOMMENDATIONS FOR DEPLOYMENT")
print("="*70)

print("""
1. THRESHOLD MANAGEMENT:
   • Current: 0.5 probability threshold
   • Could adjust based on business requirements:
     - Lower threshold (0.3): Catch more frauds, more false positives
     - Higher threshold (0.7): Fewer false positives, miss some frauds

2. MONITORING & MAINTENANCE:
   • Track model performance monthly
   • Retrain quarterly with new fraud patterns
   • Monitor for dataset drift
   • Update features if fraud tactics change

3. BUSINESS INTEGRATION:
   • Flag high-confidence frauds (>80% probability) for auto-blocking
   • Flag medium-confidence (50-80%) for manual review
   • Provide fraud reasons to customer service
   • Use explanations for customer communication

4. FEATURE SELECTION:
   • Top 10-15 features explain 80%+ of variance
   • Could reduce input features for faster prediction
   • Improves model interpretability

5. ENSEMBLE APPROACH:
   • Combine multiple models for production
   • Vote between Random Forest, SVM, Logistic Regression
   • More robust to model-specific weaknesses
""")

In [ ]:
# Save models for deployment
joblib.dump(best_model_final, 'fraud_detection_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

print("\n" + "="*70)
print("MODEL DEPLOYMENT")
print("="*70)

print("✓ Model saved as 'fraud_detection_model.pkl'")
print("✓ Scaler saved as 'feature_scaler.pkl'")

print("""
For production deployment:

```python
import joblib
import numpy as np

# Load model and scaler
model = joblib.load('fraud_detection_model.pkl')
scaler = joblib.load('feature_scaler.pkl')

# New transaction (30 features)
new_transaction = [...]

# Preprocess
new_transaction_scaled = scaler.transform([new_transaction])

# Predict
fraud_probability = model.predict_proba(new_transaction_scaled)[0, 1]

if fraud_probability > 0.7:
    print(f"⚠️  FRAUD ALERT! Confidence: {fraud_probability:.1%}")
elif fraud_probability > 0.5:
    print(f"⚠️  REVIEW RECOMMENDED (Confidence: {fraud_probability:.1%})")
else:
    print(f"✓ APPROVED (Risk: {fraud_probability:.1%})")
```
""")

## SECTION 10: SUMMARY

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════╗
║              CREDIT CARD FRAUD DETECTION PROJECT SUMMARY                  ║
╚════════════════════════════════════════════════════════════════════════════╝

✓ COMPLETED SECTIONS:
  1. Data Loading & Exploration
  2. Class Imbalance Analysis
  3. Feature Exploration & Correlation Analysis
  4. Data Preprocessing (scaling, train-test split)
  5. Model Development (4 algorithms)
  6. Model Evaluation (confusion matrix, ROC-AUC, precision-recall)
  7. Hyperparameter Tuning (GridSearchCV)
  8. Feature Importance Analysis
  9. LIME Explainability (individual prediction explanations)
  10. Deployment & Recommendations

📊 FINAL RESULTS:
""")

print(comparison_df.round(4))

print(f"""
🏆 BEST MODEL: {best_model_name}
   - F1-Score: {comparison_df.loc[best_model_name, 'F1-Score']:.4f}
   - ROC-AUC: {comparison_df.loc[best_model_name, 'ROC-AUC']:.4f}
   - Recall: {comparison_df.loc[best_model_name, 'Recall']:.4f} (catches {comparison_df.loc[best_model_name, 'Recall']*100:.1f}% of frauds)

🎯 KEY LEARNINGS:
   • Handling imbalanced data (0.17% fraud rate)
   • Importance of proper evaluation metrics
   • Feature importance for model interpretability
   • Individual prediction explanations with LIME
   • Production deployment considerations

📦 DELIVERABLES:
   • Trained model (fraud_detection_model.pkl)
   • Feature scaler (feature_scaler.pkl)
   • Feature importance analysis
   • LIME explanations for 5+ examples
   • Deployment recommendations

🚀 NEXT STEPS:
   • Monitor model performance in production
   • Retrain quarterly with new data
   • Adjust fraud threshold based on business metrics
   • Implement ensemble methods for robustness
   • Share predictions with customers using explanations
""")